# 15 · Layered 3D access maps with deck.gl

**Question:** Can an interactive map compare access scenarios without confusing people with metres?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Use Astra to critique visual encodings and Codex to implement a layer toggle while preserving computed access results.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Keep physical height and thematic height separate · 35 minutes
**Learn:** combine extruded polygons, street paths, facility markers and picking in deck.gl.
The renderer consumes plain JSON generated by Python; no ipywidgets or pydeck extension is required.
Physical mode uses building heights in metres. Population mode uses **3 display metres per resident** as a
thematic chart. This is never described as actual building height. Color uses the same 0–30 minute scale in
both access scenarios, with values above 30 clipped to the top color; exact values remain in tooltips/table.
**Predict:** which change should affect color, and which should affect extrusion?


In [ ]:
from scenes import building_features, export_scene, scene_frame
c=city(); pop=allocate(c['capacity'],1200)
nodes,graph=network(blocked=[(r*11+5,r*11+6) for r in range(10)])
idx,connector=snap(c['xy'],nodes); clinic=22; added_clinic=94
baseline=(shortest(graph,[clinic])[idx]+connector)/80.
improved=(shortest(graph,[clinic,added_clinic])[idx]+connector)/80.
assert np.isfinite(baseline).all() and (improved<=baseline+1e-9).all()
geo=building_features(c,pop)
records=[]
for i,f in enumerate(geo['features']):
    records.append(dict(**f['properties'],polygon=f['geometry']['coordinates'][0],
                        baseline=float(baseline[i]),improved=float(improved[i])))
roads=[[[float(a) for a in lonlat(*nodes[u])],[float(a) for a in lonlat(*nodes[v])]] for u in graph for v,w in graph[u] if u<v]
clinics=[dict(position=[float(v) for v in lonlat(*nodes[node])],added=node==added_clinic) for node in [clinic,added_clinic]]
payload=dict(buildings=records,roads=roads,clinics=clinics,center=[float(v) for v in lonlat(1000,1000)],synthetic=True)
export_json('15_layers.json',payload)
print('Population-weighted minutes:',float(np.average(baseline,weights=pop)),'->',float(np.average(improved,weights=pop)))


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,times,label in zip(axes,[baseline,improved],['Existing clinic','Additional clinic']):
    dots=ax.scatter(*c['xy'].T,c=times,vmin=0,vmax=30,cmap='viridis_r',s=25)
    map_axes(ax,label)
fig.colorbar(dots,ax=list(axes),label='Walking minutes (same 0–30 scale)',extend='max'); plt.show()


## Use and share the interactive scene
The scene appears below in a trusted notebook output. If your notebook viewer blocks the iframe,
download the HTML from `exports/` and open it in a browser. All data are embedded, but the pinned
JavaScript/CSS renderer still downloads from a CDN; this is not fully offline. WebGL must be available.
The static plot and numerical outputs remain usable when the renderer cannot load.

Camera rotation and vertical exaggeration change the view, not the underlying data. Keep a plan view
beside a perspective view to detect occlusion. Download the notebook, scene and evidence JSON before
clearing browser storage. Do not label synthetic geometry as observed Gaborone buildings.


In [ ]:
document=export_scene('15_deckgl_scene.html','deck',payload)
display(scene_frame(document,'deck.gl population and access scenario layers'))


## Astra critique and a Codex layer task
**Astra:** provide `15_layers.json` and screenshots of both scenarios with the same camera.
Ask: “Calculate population-weighted mean travel time for each scenario. Explain whether any building's access
worsened, and identify how the population extrusion could mislead a viewer about physical height.” Check its
numbers against this notebook, not apparent column volume. Declare a 0.01-minute tolerance before the trial.

**Codex:** “Add a population-threshold filter and a served-within-15-minutes color mode in
`content/scene_templates/deck.html`. Preserve the original arrays, keep tooltips exact, and show the visible
population separately from the whole-city total. Make the legend and table reflect the active mode.”

**Acceptance:** additional-clinic travel time never exceeds baseline; population is conserved; selecting a
display metric changes no numerical data; the same color maps to the same value in both scenarios.
Try the scene with keyboard-accessible selects and compare its table with the Python export.

References: [deck.gl standalone](https://deck.gl/docs/get-started/using-standalone) and
[PolygonLayer](https://deck.gl/docs/api-reference/layers/polygon-layer).


## Extend with Codex or Astra
Add trips or OD arcs with timestamps and explicit units. Keep a fixed color domain across animation frames, distinguish observed journeys from synthetic routes, and test that filtering does not silently change summary denominators.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
